# 🔍 Agentic Search for Context Engineering
### A Practical Implementation Guide

**Based on concepts from Leonie Monigatti's workshop — AI Engineer Europe 2026**

youtube video link : https://youtu.be/ynJyIKwjonM?si=zpOZopPsDLDegMRB

---

## 📋 What You'll Learn

| Concept | What It Means | Why It Matters |
|---------|--------------|----------------|
| **Context Engineering** | The art of curating what enters an LLM's context window | ~80% of context engineering IS search — deciding *what* to retrieve |
| **Agentic Search** | An LLM agent decides *if*, *when*, and *what* to retrieve | Beats fixed RAG pipelines that always retrieve regardless of need |
| **Tool Descriptions** | Detailed metadata about what each tool does | The #1 reason agents fail is poor tool descriptions, not poor models |
| **Agent Skills** | Dynamically injecting documentation into context | Reduces SQL/syntax errors from ~60% to ~10% |
| **Shell Tool** | Let agents run file-system commands | Most versatile but most dangerous — needs sandboxing |

## 🏗️ Architecture

```text
  ┌─────────────────────────────────────────────┐
  │                   USER QUERY                │
  └──────────────────┬──────────────────────────┘
                     │
                     ▼
  ┌─────────────────────────────────────────────┐
  │         REACT AGENT (DeepSeek LLM)          │
  │   Decides: Which tool? What parameters?     │
  └──┬──────────┬──────────┬──────────┬─────────┘
     │          │          │          │
     ▼          ▼          ▼          ▼
  ┌──────┐  ┌──────┐  ┌──────┐  ┌──────────┐
  │Vector│  │  SQL │  │Shell │  │  Agent   │
  │Search│  │Query │  │Search│  │  Skills  │
  └──┬───┘  └──┬───┘  └──┬───┘  └────┬─────┘
     │         │         │           │
     ▼         ▼         ▼           ▼
  ┌──────┐  ┌──────┐  ┌──────┐  ┌──────────┐
  │ Wiki │  │  DB  │  │Files │  │   Docs   │
  │  +   │  │      │  │      │  │ (Schema, │
  │ Docs │  │      │  │      │  │  API)    │
  └──────┘  └──────┘  └──────┘  └──────────┘
```

## 📚 From RAG → Agentic RAG → Agentic Search

```text
  Traditional RAG          Agentic RAG          Agentic Search
  ┌──────────┐          ┌──────────┐         ┌──────────┐
  │  Query   │          │  Query   │         │  Query   │
  └────┬─────┘          └────┬─────┘         └────┬─────┘
       │                     │                    │
       ▼                     ▼                    ▼
  ┌──────────┐          ┌──────────┐         ┌──────────┐
  │ ALWAYS   │          │  Agent   │         │  Agent   │
  │ retrieve │          │ decides: │         │ picks    │
  │ from     │          │ retrieve │         │ RIGHT    │
  │ ONE      │          │ or not?  │         │ tool +   │
  │ source   │          │          │         │ params   │
  └────┬─────┘          └────┬─────┘         └────┬─────┘
       │                     │                    │
       ▼                     ▼                    ▼
  ┌──────────┐          ┌──────────┐    ┌───┬───┬┴──┬──────┐
  │  Vector  │          │  Vector  │    │Vec│SQL│Shl│Skills │
  │  Store   │          │  Store   │    │   │   │   │      │
  └──────────┘          └──────────┘    └───┴───┴───┴──────┘
  
  ❌ Brittle              ⚠️ Better          ✅ Robust
  ❌ One source           ✅ Optional        ✅ Multi-source
  ❌ Always retrieves     ✅ Skips if not    ✅ Right tool, right
                          needed             params, right time
```

# Installation

In [17]:
# ===== Installation =====
# Install all required packages (quiet mode to reduce noise)
!pip install -qU \
    langchain \
    langchain-openai \
    langchain-community \
    langchain-huggingface \
    langchain-chroma \
    langgraph \
    chromadb \
    sentence-transformers \
    pydantic

print("✅ All packages installed successfully!")

✅ All packages installed successfully!


#  Imports & Configuration

In [18]:
# ===== Imports & Configuration =====

import os
import sys
import json
import sqlite3
import logging
import tempfile
import shutil
from pathlib import Path
from typing import Optional, Annotated
from datetime import datetime

# LangChain core
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.output_parsers import StrOutputParser

# LangChain integrations
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# LangGraph for the agent
from langgraph.prebuilt import create_react_agent

# Pydantic for schema validation
from pydantic import BaseModel, Field

# ChromaDB
import chromadb
import warnings
warnings.filterwarnings("ignore", message="The secret `HF_TOKEN`")
warnings.filterwarnings("ignore", category=UserWarning, module="huggingface_hub")

# ─── Logging Setup ───────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger("agentic_search")

# ─── Configuration ───────────────────────────────────────────
class Config:
    """Central configuration for the project."""

    # --- LLM Configuration ---
    # DeepSeek uses an OpenAI-compatible API
    DEEPSEEK_API_KEY: str = "put_your_key_here"
    DEEPSEEK_BASE_URL: str = "https://api.deepseek.com"
    DEEPSEEK_MODEL: str = "deepseek-chat"       # Cheapest model
    LLM_TEMPERATURE: float = 0.0                 # Deterministic for tool use

    # --- Embedding Configuration ---
    # Using a free, local embedding model (no API costs!)
    EMBEDDING_MODEL: str = "sentence-transformers/all-MiniLM-L6-v2"

    # --- Vector Store ---
    CHROMA_COLLECTION: str = "techcorp_wiki"

    # --- Data Paths ---
    DATA_DIR: Path = Path("/content/techcorp_data")
    DB_PATH: Path = Path("/content/techcorp_data/techcorp.db")

    # --- Search Defaults ---
    DEFAULT_TOP_K: int = 3

config = Config()

# ─── Try loading API key from Colab secrets (safer) ──────────
try:
    from google.colab import userdata
    config.DEEPSEEK_API_KEY = userdata.get("DEEPSEEK_API_KEY")
    logger.info("✅ API key loaded from Colab secrets")
except Exception:
    logger.warning("⚠️ Using hardcoded API key. For production, use Colab secrets or .env files.")

# ─── Initialize LLM ─────────────────────────────────────────
llm = ChatOpenAI(
    model=config.DEEPSEEK_MODEL,
    api_key=config.DEEPSEEK_API_KEY,
    base_url=config.DEEPSEEK_BASE_URL,
    temperature=config.LLM_TEMPERATURE,
    max_tokens=1024,   # Limit output to save costs
)

# ─── Initialize Embeddings (free, local) ────────────────────
embeddings = HuggingFaceEmbeddings(
    model_name=config.EMBEDDING_MODEL,
    model_kwargs={"device": "cpu"},   # Colab CPU is fine for this
    encode_kwargs={"normalize_embeddings": True},
)

logger.info("✅ LLM and Embeddings initialized!")
logger.info(f"   Model: {config.DEEPSEEK_MODEL}")
logger.info(f"   Embeddings: {config.EMBEDDING_MODEL}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## 🧠 Section 1: Understanding Context Engineering

### What is Context Engineering?

**Context Engineering** is the art and science of deciding **what information goes into an LLM's context window** before it generates a response.

Think of it this way:
- **Prompt Engineering** = How you phrase your question
- **Context Engineering** = What background material you provide alongside your question

**Example:**
```text
Without context engineering:
  User: "What's our refund policy?"
  LLM: "I don't have access to your company's refund policy."

With context engineering:
  User: "What's our refund policy?"
  → Agent searches wiki → finds refund policy doc → injects it
  LLM: "According to the company wiki, refunds are processed within 14 days..."
```

### Why is Search 80% of Context Engineering?

Because the LLM already knows *how* to answer — it just needs the *right information*. The hard part is:
1. **Knowing WHEN to search** (not every question needs retrieval)
2. **Knowing WHERE to search** (database? files? web? wiki?)
3. **Knowing HOW to search** (what query? what parameters?)
4. **Knowing WHAT to return** (full doc? summary? specific field?)

### From RAG to Agentic RAG to Agentic Search

| Stage | How It Works | Problem |
|-------|-------------|---------|
| **Naive RAG** | Always retrieve from vector store, then generate | Irrelevant retrievals pollute context |
| **Agentic RAG** | Agent decides whether to retrieve or not | Only one search tool = limited |
| **Agentic Search** | Agent picks the RIGHT tool from a stack | Combines semantic, SQL, shell, skills |

### The Four Failure Modes of Agentic Search

1. **Not calling a tool** — Agent tries to answer from memory (hallucination)
2. **Calling the wrong tool** — Uses semantic search for structured data
3. **Generating wrong parameters** — Bad SQL, wrong file path, etc.
4. **Calling too many tools** — Wastes tokens and adds noise

**The #1 fix for all of these? Better tool descriptions.** That's what we'll demonstrate.

# Data Foundation Setup

In [19]:
# ===== Data Foundation Setup =====
# We create a fictional company "TechCorp" with data across 4 sources:
# 1. Wiki documents (for semantic search)
# 2. SQLite database (for SQL queries)
# 3. File system (for shell/file search)
# 4. API documentation (for agent skills)

# ─── Clean up any previous runs ─────────────────────────────
if config.DATA_DIR.exists():
    shutil.rmtree(config.DATA_DIR)
config.DATA_DIR.mkdir(parents=True, exist_ok=True)

# ═══════════════════════════════════════════════════════════════
# SOURCE 1: Wiki / Knowledge Base Documents
# These will be embedded and stored in ChromaDB for semantic search
# ═══════════════════════════════════════════════════════════════

wiki_documents = [
    {
        "id": "wiki_001",
        "title": "Remote Work Policy",
        "content": """TechCorp Remote Work Policy (Updated 2025)

Eligibility: All full-time employees who have completed their 90-day probation period are eligible for remote work.

Schedule: Employees may work remotely up to 3 days per week. At least 2 days must be in-office for team collaboration. Monday and Friday are the most popular remote days.

Equipment: TechCorp provides a $1,500 home office stipend for new remote workers. This covers desk, chair, monitor, and peripherals. Submit receipts through the HR portal within 90 days.

Connectivity: Minimum 50 Mbps internet connection required. TechCorp reimburses up to $75/month for internet costs.

Security: All remote workers must use the company VPN (CloudGuard) when accessing internal systems. MFA is mandatory. Never store sensitive data on personal devices.

Time Tracking: Use TimelyApp to log hours. Remote days must be marked in the calendar by 9 AM."""
    },
    {
        "id": "wiki_002",
        "title": "Leave and PTO Policy",
        "content": """TechCorp Leave and PTO Policy (Updated 2025)

Annual Leave: Full-time employees receive 20 PTO days per year, accrued at 1.67 days per month. PTO rolls over up to 5 days into the next year.

Sick Leave: 10 paid sick days per year. No doctor's note required for absences under 3 consecutive days.

Parental Leave: 16 weeks paid parental leave for primary caregivers, 6 weeks for secondary caregivers. Must be taken within 12 months of birth or adoption.

Bereavement Leave: 5 days for immediate family, 3 days for extended family.

Sabbatical: After 5 years of continuous employment, employees are eligible for a 4-week paid sabbatical. Requires VP approval and 3-month advance notice.

Holiday Schedule: TechCorp observes 11 paid holidays per year, including the week between Christmas and New Year's Day."""
    },
    {
        "id": "wiki_003",
        "title": "API Development Standards",
        "content": """TechCorp API Development Standards (v3.2)

Architecture: All new APIs must follow RESTful principles. Use JSON for request/response bodies. Version all APIs using URI path versioning (e.g., /api/v1/resource).

Authentication: All APIs must use OAuth 2.0 with JWT tokens. Token expiry: 1 hour for access tokens, 7 days for refresh tokens. Use the Auth0 integration for token management.

Error Handling: Use standard HTTP status codes. Return errors in the format:
{"error": {"code": "ERROR_CODE", "message": "Human-readable message", "details": {}}}

Rate Limiting: Default rate limit is 100 requests per minute per API key. Return 429 status code with Retry-After header when exceeded.

Naming Conventions: Use plural nouns for resources (e.g., /users, not /user). Use camelCase for JSON field names. Use kebab-case for URL paths.

Documentation: All endpoints must have OpenAPI 3.0 specifications. Use Swagger UI for interactive documentation. Include request/response examples."""
    },
    {
        "id": "wiki_004",
        "title": "Onboarding Process",
        "content": """TechCorp Onboarding Process (Updated 2025)

Day 1: Welcome session at 9 AM in the main conference room. Receive laptop and badge. Complete IT setup with Help Desk (Room 204). Lunch with team at 12 PM.

Week 1: Complete mandatory compliance training (4 modules, ~8 hours total). Set up development environment following the Engineering Setup Guide. Meet with your onboarding buddy daily.

Month 1: Complete 30-day check-in with manager. Review initial goals and expectations. Attend "TechCorp 101" session covering company history and culture.

Month 3: 90-day review with manager and HR. Formal transition from probation to full employment. Submit remote work request if desired.

IT Setup Steps:
1. Claim your laptop from IT (Room 204)
2. Log in with your temporary credentials (emailed to you)
3. Change password within 24 hours
4. Install CloudGuard VPN from the software center
5. Set up TimelyApp for time tracking
6. Join Slack workspace (invitation sent to your email)"""
    },
    {
        "id": "wiki_005",
        "title": "Security Incident Response",
        "content": """TechCorp Security Incident Response Plan (v2.1)

Severity Levels:
- P1 (Critical): Data breach, ransomware, complete service outage. Response time: 15 minutes.
- P2 (High): Partial service degradation, suspected unauthorized access. Response time: 1 hour.
- P3 (Medium): Vulnerability disclosure, non-sensitive data exposure. Response time: 4 hours.
- P4 (Low): Policy violation, suspicious but unconfirmed activity. Response time: 24 hours.

Escalation Path:
1. Report incident to security@techcorp.com or #security-incidents on Slack
2. On-call security engineer triages within response time
3. P1/P2: Incident commander assigned within 30 minutes
4. All incidents: Root cause analysis within 48 hours

Key Contacts:
- CISO: Sarah Chen (sarah.chen@techcorp.com)
- Security Lead: Marcus Rivera (marcus.rivera@techcorp.com)
- On-call rotation: Check #security-oncall on Slack

Post-Incident: Blameless retrospective within 5 business days. Action items tracked in Jira with 30-day completion deadline."""
    },
]

# ═══════════════════════════════════════════════════════════════
# SOURCE 2: SQLite Database — Employee & Project Data
# ═══════════════════════════════════════════════════════════════

conn = sqlite3.connect(str(config.DB_PATH))
cursor = conn.cursor()

# Create tables
cursor.execute("""
CREATE TABLE IF NOT EXISTS departments (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    budget REAL,
    head_count INTEGER
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS employees (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    department TEXT NOT NULL,
    role TEXT NOT NULL,
    email TEXT NOT NULL,
    hire_date TEXT NOT NULL,
    salary INTEGER
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS projects (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    department_id INTEGER,
    status TEXT NOT NULL,
    start_date TEXT NOT NULL,
    end_date TEXT,
    budget REAL,
    FOREIGN KEY (department_id) REFERENCES departments(id)
)
""")

# Insert sample data
departments_data = [
    (1, "Engineering", 2500000, 45),
    (2, "Marketing", 800000, 18),
    (3, "Sales", 1200000, 22),
    (4, "Human Resources", 400000, 8),
    (5, "Data Science", 900000, 15),
]

employees_data = [
    (1, "Alice Johnson", "Engineering", "Senior Backend Engineer", "alice.johnson@techcorp.com", "2021-03-15", 145000),
    (2, "Bob Smith", "Engineering", "DevOps Lead", "bob.smith@techcorp.com", "2020-08-22", 155000),
    (3, "Carol Williams", "Marketing", "Content Strategist", "carol.williams@techcorp.com", "2022-01-10", 95000),
    (4, "David Brown", "Sales", "Account Executive", "david.brown@techcorp.com", "2023-06-01", 85000),
    (5, "Eve Davis", "Data Science", "ML Engineer", "eve.davis@techcorp.com", "2022-09-15", 140000),
    (6, "Frank Miller", "Engineering", "Frontend Engineer", "frank.miller@techcorp.com", "2023-02-28", 120000),
    (7, "Grace Wilson", "Human Resources", "HR Business Partner", "grace.wilson@techcorp.com", "2019-11-05", 105000),
    (8, "Henry Taylor", "Sales", "Sales Manager", "henry.taylor@techcorp.com", "2021-07-20", 130000),
    (9, "Iris Anderson", "Data Science", "Data Analyst", "iris.anderson@techcorp.com", "2024-01-08", 95000),
    (10, "Jack Thomas", "Engineering", "Tech Lead", "jack.thomas@techcorp.com", "2018-05-14", 170000),
    (11, "Karen Jackson", "Marketing", "Brand Manager", "karen.jackson@techcorp.com", "2022-04-18", 110000),
    (12, "Leo White", "Engineering", "Security Engineer", "leo.white@techcorp.com", "2023-11-01", 150000),
]

projects_data = [
    (1, "CloudGuard 2.0", 1, "In Progress", "2025-01-15", "2025-09-30", 500000),
    (2, "Marketing Automation Platform", 2, "In Progress", "2025-03-01", "2025-12-31", 200000),
    (3, "Sales Pipeline Dashboard", 3, "Completed", "2024-06-01", "2025-02-28", 150000),
    (4, "Employee Wellness App", 4, "Planning", "2025-07-01", "2026-03-31", 80000),
    (5, "ML Recommendation Engine", 5, "In Progress", "2025-02-01", "2025-11-30", 350000),
    (6, "API Gateway Redesign", 1, "In Progress", "2025-04-15", "2026-01-31", 300000),
    (7, "Customer 360 Analytics", 5, "Planning", "2025-08-01", "2026-06-30", 400000),
]

cursor.executemany("INSERT OR REPLACE INTO departments VALUES (?, ?, ?, ?)", departments_data)
cursor.executemany("INSERT OR REPLACE INTO employees VALUES (?, ?, ?, ?, ?, ?, ?)", employees_data)
cursor.executemany("INSERT OR REPLACE INTO projects VALUES (?, ?, ?, ?, ?, ?, ?)", projects_data)
conn.commit()
conn.close()

# ═══════════════════════════════════════════════════════════════
# SOURCE 3: File System — Logs, Configs, and Notes
# ═══════════════════════════════════════════════════════════════

files_dir = config.DATA_DIR / "files"
files_dir.mkdir(exist_ok=True)

# Application logs
log_dir = files_dir / "logs"
log_dir.mkdir(exist_ok=True)

(log_dir / "app_2025-05-01.log").write_text("""2025-05-01 08:12:33 [INFO] Application started successfully
2025-05-01 08:12:34 [INFO] Connected to database: techcorp_prod
2025-05-01 08:15:22 [WARN] Slow query detected: SELECT * FROM employees WHERE salary > 150000 (2.3s)
2025-05-01 08:23:41 [INFO] User alice.johnson@techcorp.com logged in
2025-05-01 08:45:11 [ERROR] Failed to send email notification: SMTP connection timeout
2025-05-01 08:45:12 [INFO] Retrying email notification in 30 seconds...
2025-05-01 08:45:42 [INFO] Email notification sent successfully on retry
2025-05-01 09:01:15 [WARN] API rate limit approaching for key: prod_api_key_7829 (87/100 requests)
2025-05-01 09:12:33 [INFO] Scheduled backup completed: backup_20250501.sql (45.2 MB)
""")

(log_dir / "app_2025-05-02.log").write_text("""2025-05-02 07:55:01 [INFO] Application started successfully
2025-05-02 07:55:02 [INFO] Connected to database: techcorp_prod
2025-05-02 08:03:22 [ERROR] Database connection pool exhausted. Active connections: 50/50
2025-05-02 08:03:23 [WARN] Request queued: GET /api/v1/employees (waiting for connection)
2025-05-02 08:05:44 [INFO] Connection pool recovered. Released 15 idle connections.
2025-05-02 08:30:00 [INFO] Monthly report generation started
2025-05-02 08:31:22 [INFO] Monthly report generated: may_2025_report.pdf
2025-05-02 09:15:33 [ERROR] Payment gateway timeout: Stripe API returned 504
2025-05-02 09:15:34 [WARN] Payment processing moved to retry queue
2025-05-02 09:20:11 [INFO] Payment processed successfully on retry: INV-2025-0452
""")

(log_dir / "error_2025-05-02.log").write_text("""2025-05-02 08:03:22 [CRITICAL] Database connection pool exhausted
  at ConnectionPool.acquire(ConnectionPool.java:142)
  at DatabaseService.query(DatabaseService.java:89)
  at EmployeeAPI.getEmployees(EmployeeAPI.java:56)
  Root cause: Too many concurrent requests during morning login rush
  Affected users: ~200 employees attempting to access the portal
  Resolution: Increased pool size from 50 to 75. Added connection timeout of 30s.

2025-05-02 09:15:33 [CRITICAL] Payment gateway timeout
  at PaymentService.processPayment(PaymentService.java:234)
  at InvoiceController.pay(InvoiceController.java:78)
  Root cause: Stripe API degradation (status.stripe.com incident #INC-44521)
  Affected transactions: 3 payments totaling $12,450
  Resolution: Implemented exponential backoff retry. All payments recovered.
""")

# Configuration files
config_dir = files_dir / "config"
config_dir.mkdir(exist_ok=True)

(config_dir / "app_config.json").write_text(json.dumps({
    "app_name": "TechCorp Portal",
    "version": "3.2.1",
    "database": {
        "host": "db.internal.techcorp.com",
        "port": 5432,
        "name": "techcorp_prod",
        "pool_size": 75,
        "timeout_seconds": 30
    },
    "api": {
        "rate_limit_per_minute": 100,
        "max_request_size_mb": 10,
        "timeout_seconds": 60
    },
    "email": {
        "smtp_host": "smtp.techcorp.com",
        "smtp_port": 587,
        "from_address": "noreply@techcorp.com"
    },
    "features": {
        "remote_work_requests": True,
        "pto_requests": True,
        "expense_reports": True,
        "dark_mode": True
    }
}, indent=2))

(config_dir / "deployment_config.yaml").write_text("""# TechCorp Portal Deployment Configuration
environment: production
region: us-east-1
instances:
  - type: t3.xlarge
    count: 3
    purpose: web-servers
  - type: r5.large
    count: 2
    purpose: database-replicas
  - type: cache.r5.large
    count: 1
    purpose: redis-cache

scaling:
  min_instances: 3
  max_instances: 10
  target_cpu_percent: 70

monitoring:
  health_check_interval: 30s
  alert_email: ops@techcorp.com
  pager_duty: techcorp-prod
""")

# Meeting notes
notes_dir = files_dir / "notes"
notes_dir.mkdir(exist_ok=True)

(notes_dir / "q2_planning_meeting.txt").write_text("""Q2 2025 Planning Meeting — April 15, 2025
Attendees: Jack Thomas, Alice Johnson, Henry Taylor, Grace Wilson

Key Decisions:
1. CloudGuard 2.0 MVP target: August 2025
2. New hire plan: 5 engineers for API Gateway team (Q2), 3 data scientists (Q3)
3. Marketing budget increased by 15% for Q2-Q3 campaigns
4. Employee wellness app development begins Q3 2025

Action Items:
- Jack: Finalize CloudGuard 2.0 technical spec by April 30
- Alice: Review API Gateway architecture proposal by May 15
- Henry: Prepare Q1 sales report for board meeting
- Grace: Begin recruiting for 5 engineering positions

Budget Allocations:
- CloudGuard 2.0: $500K (approved)
- API Gateway Redesign: $300K (approved)
- ML Recommendation Engine: $350K (approved)
- New hires (5 engineers): ~$875K annually
""")

(notes_dir / "incident_postmortem_2025-04-28.txt").write_text("""Incident Postmortem — Database Connection Pool Exhaustion
Date: April 28, 2025
Severity: P2
Duration: 2 hours 15 minutes (6:00 AM - 8:15 AM UTC)

Summary:
The employee portal became unresponsive during the morning login rush.
The database connection pool (50 connections) was exhausted by concurrent
requests, causing all new requests to queue and timeout.

Root Cause:
- Morning login rush generated 3x normal traffic between 6-8 AM UTC
- Connection pool was set to 50 (configured for average, not peak load)
- No connection timeout was configured, so idle connections were never released
- Monitoring alert for pool usage was set at 90% instead of 70%

Fixes Applied:
1. Increased pool size from 50 to 75 connections
2. Added connection timeout of 30 seconds
3. Lowered monitoring alert threshold to 70%
4. Implemented connection pool metrics dashboard

Lessons Learned:
- Always configure for peak load, not average
- Set connection timeouts to prevent idle connection hoarding
- Monitor resource pools proactively, not reactively
""")

# ═══════════════════════════════════════════════════════════════
# SOURCE 4: API Documentation (for Agent Skills)
# ═══════════════════════════════════════════════════════════════

api_docs_dir = config.DATA_DIR / "api_docs"
api_docs_dir.mkdir(exist_ok=True)

(api_docs_dir / "employee_api.md").write_text("""# Employee API Documentation (v2)

## Base URL
`/api/v2/employees`

## Endpoints

### List Employees
`GET /api/v2/employees`

Query Parameters:
- `department` (string, optional): Filter by department name
- `role` (string, optional): Filter by role title
- `min_salary` (integer, optional): Minimum salary filter
- `max_salary` (integer, optional): Maximum salary filter
- `hire_date_after` (string, optional): Filter by hire date (YYYY-MM-DD format)
- `page` (integer, optional): Page number (default: 1)
- `per_page` (integer, optional): Results per page (default: 20, max: 100)

Example:
GET /api/v2/employees?department=Engineering&min_salary=140000


### Get Employee by ID
`GET /api/v2/employees/{id}`

Returns a single employee object with all fields.

### Create Employee
`POST /api/v2/employees`

Required fields: name, department, role, email, hire_date, salary

### Update Employee
`PATCH /api/v2/employees/{id}`

Updatable fields: department, role, salary, email

### Delete Employee
`DELETE /api/v2/employees/{id}`

Soft delete — employee record is marked inactive.
""")

print("✅ Data foundation created!")
print(f"   📄 Wiki documents: {len(wiki_documents)}")
print(f"   🗄️ SQLite database: {config.DB_PATH}")
print(f"   📁 File system: {files_dir}")
print(f"   📚 API docs: {api_docs_dir}")

# Quick verification
conn = sqlite3.connect(str(config.DB_PATH))
cursor = conn.cursor()
cursor.execute("SELECT COUNT(*) FROM employees")
emp_count = cursor.fetchone()[0]
cursor.execute("SELECT COUNT(*) FROM departments")
dept_count = cursor.fetchone()[0]
cursor.execute("SELECT COUNT(*) FROM projects")
proj_count = cursor.fetchone()[0]
conn.close()
print(f"\n   Database stats: {emp_count} employees, {dept_count} departments, {proj_count} projects")

# Count files
all_files = list(files_dir.rglob("*"))
print(f"   File system: {len([f for f in all_files if f.is_file()])} files across {len([f for f in all_files if f.is_dir()])} directories")

✅ Data foundation created!
   📄 Wiki documents: 5
   🗄️ SQLite database: /content/techcorp_data/techcorp.db
   📁 File system: /content/techcorp_data/files
   📚 API docs: /content/techcorp_data/api_docs

   Database stats: 12 employees, 5 departments, 7 projects
   File system: 7 files across 3 directories


# Build Vector Store (Semantic Search Index)




In [20]:
# ===== Build Vector Store =====
#
# 🧠 CONCEPT: Semantic Search
#
# Semantic search converts text into vector embeddings (arrays of numbers).
# Similar meanings → similar vectors → nearby in vector space.
#
# How it works:
# 1. Split documents into chunks
# 2. Convert each chunk into a vector using an embedding model
# 3. Store vectors in a vector database (ChromaDB)
# 4. When searching, embed the query and find nearest vectors
#
# ✅ Great for: Natural language queries, conceptual search, "what's our policy on X?"
# ❌ Bad for: Exact matches, aggregations, structured queries, latest data

from langchain_core.documents import Document

# ─── Create LangChain Documents ──────────────────────────────
documents = []
for doc in wiki_documents:
    documents.append(Document(
        page_content=doc["content"],
        metadata={
            "source": "wiki",
            "title": doc["title"],
            "doc_id": doc["id"],
        }
    ))

# ─── Build ChromaDB Vector Store ─────────────────────────────
# Using in-memory ChromaDB (no persistence needed for demo)
chroma_client = chromadb.Client()

try:
    chroma_client.delete_collection(config.CHROMA_COLLECTION)
except Exception:
    pass

vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    client=chroma_client,
    collection_name=config.CHROMA_COLLECTION,
)

# ─── Quick Test ──────────────────────────────────────────────
test_results = vectorstore.similarity_search("remote work policy", k=2)
print("🔍 Quick test — 'remote work policy':")
for i, doc in enumerate(test_results):
    print(f"\n  Result {i+1}: [{doc.metadata['title']}]")
    print(f"  Preview: {doc.page_content[:100]}...")

print("\n✅ Vector store built with", len(documents), "documents!")

🔍 Quick test — 'remote work policy':

  Result 1: [Remote Work Policy]
  Preview: TechCorp Remote Work Policy (Updated 2025)

Eligibility: All full-time employees who have completed ...

  Result 2: [Onboarding Process]
  Preview: TechCorp Onboarding Process (Updated 2025)

Day 1: Welcome session at 9 AM in the main conference ro...

✅ Vector store built with 5 documents!


## 🔧 Section 2: Building Search Tools (The Right Way)

### Why Tool Descriptions Are Everything

When you give an LLM a tool, it only knows what the **description** tells it. The LLM doesn't read your code. It doesn't know the database schema. It doesn't know what "search" means in your context.

**The workshop's key insight:** Most agent failures come from poor tool descriptions, not poor models.

### Anatomy of a Good Tool Description

```python
# ❌ BAD — Minimal description (common mistake)
@tool
def search(query: str) -> str:
    """Search the database."""
    ...

# ✅ GOOD — Detailed description
@tool
def semantic_search(query: str, top_k: int = 3) -> str:
    """Search the company wiki and knowledge base using semantic similarity.
    
    Use this tool when the user asks about company policies, procedures,
    or general knowledge that would be found in documentation.
    
    DO NOT use this tool for:
    - Looking up specific employees or departments (use query_database instead)
    - Searching log files or configs (use search_files instead)
    - Questions about API syntax (use load_api_docs instead)
    
    The query should be a natural language question or search terms.
    """
    ...
```

### Key Elements of a Good Tool Description

1. **What it does** — Clear, specific description
2. **When to use it** — Trigger conditions
3. **When NOT to use it** — Boundaries and relationships to other tools
4. **Parameter guidance** — What format/values to pass
5. **Return format** — What the tool returns

In [21]:

# ===== Tool 1 — Semantic Search =====
#
# 🔎 What: Search company wiki/docs using vector similarity
# 🎯 When: User asks about policies, procedures, general company knowledge
# 🚫 Not for: Structured data (employees, projects), files, API docs

class SemanticSearchInput(BaseModel):
    """Input schema for semantic search tool.

    Pydantic models provide:
    - Automatic validation (e.g., top_k must be 1-20)
    - Clear schema for the LLM to understand
    - Self-documenting parameters via Field descriptions
    """
    query: str = Field(
        description=(
            "A natural language search query. Should capture the intent of "
            "what the user wants to find in the company wiki. "
            "Examples: 'remote work policy', 'how to request PTO', "
            "'security incident reporting procedure'"
        )
    )
    top_k: int = Field(
        default=3,
        description="Number of results to return. Use 1-3 for specific queries, 5-10 for broad exploration.",
        ge=1,
        le=10,
    )


@tool(args_schema=SemanticSearchInput)
def semantic_search(query: str, top_k: int = 3) -> str:
    """Search the company wiki and knowledge base using semantic similarity.

    Use this tool when the user asks about company policies, procedures,
    standards, or general knowledge documented in the company wiki.

    This tool finds documents that are SEMANTICALLY SIMILAR to the query —
    it understands meaning, not just keywords.

    ✅ USE FOR:
    - "What is the remote work policy?"
    - "How do I report a security incident?"
    - "What are the API development standards?"
    - "How does onboarding work?"

    ❌ DO NOT USE FOR:
    - Looking up specific employees or departments → use query_database instead
    - Searching log files or config files → use search_files instead
    - Questions about API endpoint syntax → use load_api_docs instead
    - Counting, aggregating, or filtering structured data → use query_database instead

    Args:
        query: Natural language search query.
        top_k: Number of results to return (1-10).

    Returns:
        Formatted search results with document titles and content.
    """
    try:
        results = vectorstore.similarity_search(query, k=top_k)
        if not results:
            return "No relevant documents found in the company wiki."

        output_parts = []
        for i, doc in enumerate(results, 1):
            output_parts.append(
                f"📄 Result {i} [{doc.metadata.get('title', 'Untitled')}]:\n{doc.page_content}"
            )
        return "\n\n---\n\n".join(output_parts)

    except Exception as e:
        return f"Error during semantic search: {str(e)}"


# ─── Quick Test ──────────────────────────────────────────────
print("🧪 Testing semantic_search tool...")
result = semantic_search.invoke({"query": "What is the remote work policy?", "top_k": 2})
print(result[:500] + "\n...")
print("\n✅ Semantic search tool ready!")

🧪 Testing semantic_search tool...
📄 Result 1 [Remote Work Policy]:
TechCorp Remote Work Policy (Updated 2025)

Eligibility: All full-time employees who have completed their 90-day probation period are eligible for remote work.

Schedule: Employees may work remotely up to 3 days per week. At least 2 days must be in-office for team collaboration. Monday and Friday are the most popular remote days.

Equipment: TechCorp provides a $1,500 home office stipend for new remote workers. This covers desk, chair, monitor, and peripherals. S
...

✅ Semantic search tool ready!


# Tool 2 — Database Query

In [22]:
# ===== Tool 2 — Database Query =====
#
# 🔎 What: Execute SQL queries on the TechCorp database
# 🎯 When: User asks about employees, departments, projects, salaries, etc.
# 🚫 Not for: Company policies (wiki), files (shell), general knowledge
#
# 💡 KEY INSIGHT: This is where Agent Skills shine!
#    Without the schema, the LLM guesses table/column names → errors
#    With the schema loaded, it writes correct SQL → success

# ─── The Database Schema (this IS the Agent Skill content) ───
DB_SCHEMA = """
TechCorp Database Schema:

Table: departments
  - id (INTEGER, PRIMARY KEY)
  - name (TEXT) — Department name: Engineering, Marketing, Sales, Human Resources, Data Science
  - budget (REAL) — Annual budget in USD
  - head_count (INTEGER) — Number of employees

Table: employees
  - id (INTEGER, PRIMARY KEY)
  - name (TEXT) — Full name
  - department (TEXT) — Department name (matches departments.name)
  - role (TEXT) — Job title
  - email (TEXT) — Corporate email
  - hire_date (TEXT) — Date hired (YYYY-MM-DD format)
  - salary (INTEGER) — Annual salary in USD

Table: projects
  - id (INTEGER, PRIMARY KEY)
  - name (TEXT) — Project name
  - department_id (INTEGER, FOREIGN KEY → departments.id)
  - status (TEXT) — One of: Planning, In Progress, Completed
  - start_date (TEXT) — Start date (YYYY-MM-DD format)
  - end_date (TEXT) — End date (YYYY-MM-DD format, NULL if not completed)
  - budget (REAL) — Project budget in USD

Relationships:
  - employees.department → departments.name
  - projects.department_id → departments.id

Common query patterns:
  - Employees in a department: SELECT * FROM employees WHERE department = 'Engineering'
  - Department stats: SELECT department, COUNT(*), AVG(salary) FROM employees GROUP BY department
  - Active projects: SELECT * FROM projects WHERE status = 'In Progress'
  - High earners: SELECT name, salary FROM employees WHERE salary > 140000 ORDER BY salary DESC
"""

class DatabaseQueryInput(BaseModel):
    """Input schema for database query tool."""
    sql_query: str = Field(
        description=(
            "A valid SQL query to execute against the TechCorp database. "
            "Only SELECT queries are allowed. "
            "The database has tables: employees, departments, projects. "
            "IMPORTANT: Use the load_db_schema tool first if you're unsure about table names or columns."
        )
    )


@tool(args_schema=DatabaseQueryInput)
def query_database(sql_query: str) -> str:
    """Execute a SQL query on the TechCorp employee/project database.

    Use this tool when the user asks about specific employees, departments,
    projects, salaries, or any STRUCTURED DATA that requires filtering,
    counting, or aggregating.

    ✅ USE FOR:
    - "Who are the engineers earning over 140k?"
    - "How many employees are in each department?"
    - "What projects are currently in progress?"
    - "Who was hired most recently in Marketing?"
    - "What is the average salary by department?"

    ❌ DO NOT USE FOR:
    - Company policies or procedures → use semantic_search instead
    - Searching log files → use search_files instead
    - API documentation → use load_api_docs instead

    ⚠️ IMPORTANT: Only SELECT queries are allowed. No INSERT, UPDATE, or DELETE.

    If you're unsure about table or column names, call load_db_schema first
    to get the complete database schema.

    Args:
        sql_query: A valid SQL SELECT query.

    Returns:
        Query results formatted as a table, or an error message.
    """
    # Safety: Block destructive queries
    forbidden_keywords = ["INSERT", "UPDATE", "DELETE", "DROP", "ALTER", "CREATE", "TRUNCATE"]
    query_upper = sql_query.upper().strip()
    for keyword in forbidden_keywords:
        if keyword in query_upper:
            return f"❌ Error: {keyword} operations are not allowed. Only SELECT queries are permitted."

    try:
        conn = sqlite3.connect(str(config.DB_PATH))
        cursor = conn.cursor()
        cursor.execute(sql_query)

        # Get column names
        columns = [description[0] for description in cursor.description]
        rows = cursor.fetchall()
        conn.close()

        if not rows:
            return "Query returned 0 results."

        # Format as a readable table
        result_lines = []
        # Header
        result_lines.append(" | ".join(columns))
        result_lines.append("-" * (len(" | ".join(columns))))
        # Rows
        for row in rows:
            result_lines.append(" | ".join(str(val) for val in row))

        summary = f"Query returned {len(rows)} row(s):\n\n" + "\n".join(result_lines)
        return summary

    except Exception as e:
        return f"❌ SQL Error: {str(e)}\n\nTip: Use load_db_schema to review the database schema before writing queries."


# ─── Quick Test ──────────────────────────────────────────────
print("🧪 Testing query_database tool...")
result = query_database.invoke({"sql_query": "SELECT name, role, salary FROM employees WHERE department = 'Engineering' ORDER BY salary DESC"})
print(result)
print("\n✅ Database query tool ready!")

🧪 Testing query_database tool...
Query returned 5 row(s):

name | role | salary
--------------------
Jack Thomas | Tech Lead | 170000
Bob Smith | DevOps Lead | 155000
Leo White | Security Engineer | 150000
Alice Johnson | Senior Backend Engineer | 145000
Frank Miller | Frontend Engineer | 120000

✅ Database query tool ready!


# Tool 3 — File System Search

In [23]:
# ===== Tool 3 — File System Search =====
#
# 🔎 What: Search and read files from the TechCorp file system
# 🎯 When: User asks about logs, configs, meeting notes, or files
# 🚫 Not for: Structured data (DB), policies (wiki), API syntax (docs)
#
# 💡 This is the "Shell Tool" concept from the workshop — but SAFER.
#    Instead of giving the agent a raw shell, we provide specific operations:
#    - List files in a directory
#    - Read file contents
#    - Search for text patterns (grep-like)
#    This is the "sandboxed shell" approach recommended in the lecture.

class FileSearchInput(BaseModel):
    """Input schema for file system search tool."""
    operation: str = Field(
        description=(
            "The file operation to perform. Must be one of: "
            "'list' (list files in a directory), "
            "'read' (read contents of a specific file), "
            "'grep' (search for a text pattern across all files)."
        )
    )
    path: str = Field(
        default="",
        description=(
            "File or directory path relative to the data root. "
            "For 'list': directory path (e.g., 'logs', 'config', 'notes'). "
            "For 'read': file path (e.g., 'logs/app_2025-05-01.log'). "
            "For 'grep': not needed (searches all files)."
        )
    )
    pattern: str = Field(
        default="",
        description=(
            "Search pattern for 'grep' operation. "
            "Case-insensitive text search across all files. "
            "Examples: 'ERROR', 'timeout', 'database', 'Alice'"
        )
    )


@tool(args_schema=FileSearchInput)
def search_files(operation: str, path: str = "", pattern: str = "") -> str:
    """Search and read files from the TechCorp file system.

    Use this tool to access log files, configuration files, meeting notes,
    and other documents stored on the file system.

    ✅ USE FOR:
    - "Check the application logs for errors"
    - "What's in the app config file?"
    - "Find all files mentioning 'database timeout'"
    - "Read the Q2 planning meeting notes"
    - "Show me the deployment configuration"

    ❌ DO NOT USE FOR:
    - Company policies → use semantic_search instead
    - Employee/project data → use query_database instead
    - API documentation → use load_api_docs instead

    OPERATIONS:
    - 'list': List all files in a directory. Provide 'path' (e.g., 'logs', 'config', 'notes').
    - 'read': Read contents of a file. Provide 'path' (e.g., 'logs/app_2025-05-01.log').
    - 'grep': Search for a text pattern across ALL files. Provide 'pattern'.

    SAFETY: This tool only provides read access. No write, delete, or execute operations.
    All paths are restricted to the TechCorp data directory.

    Args:
        operation: One of 'list', 'read', or 'grep'.
        path: File or directory path (for 'list' and 'read').
        pattern: Text search pattern (for 'grep').

    Returns:
        File listing, file contents, or grep results.
    """
    base_dir = config.DATA_DIR / "files"

    # Security: Prevent path traversal
    try:
        if path:
            full_path = (base_dir / path).resolve()
            base_resolved = base_dir.resolve()
            if not str(full_path).startswith(str(base_resolved)):
                return "❌ Error: Access denied. Path is outside the allowed directory."
    except Exception:
        return "❌ Error: Invalid path."

    if operation == "list":
        target_dir = base_dir / path if path else base_dir
        if not target_dir.is_dir():
            return f"❌ Directory not found: {path}"

        items = sorted(target_dir.iterdir())
        if not items:
            return f"Directory '{path}' is empty."

        output_lines = [f"📂 {path or 'root'}:"]
        for item in items:
            if item.is_dir():
                sub_files = len(list(item.iterdir()))
                output_lines.append(f"  📁 {item.name}/ ({sub_files} items)")
            else:
                size = item.stat().st_size
                output_lines.append(f"  📄 {item.name} ({size:,} bytes)")
        return "\n".join(output_lines)

    elif operation == "read":
        if not path:
            return "❌ Please provide a file path for the 'read' operation."
        file_path = base_dir / path
        if not file_path.is_file():
            return f"❌ File not found: {path}. Use 'list' to see available files."

        content = file_path.read_text()
        # Truncate very long files
        max_chars = 3000
        if len(content) > max_chars:
            content = content[:max_chars] + f"\n\n... [truncated, {len(content):,} total characters]"
        return f"📄 {path}:\n\n{content}"

    elif operation == "grep":
        if not pattern:
            return "❌ Please provide a search pattern for the 'grep' operation."

        results = []
        for file_path in sorted(base_dir.rglob("*")):
            if not file_path.is_file():
                continue
            try:
                content = file_path.read_text()
                rel_path = file_path.relative_to(base_dir)
                for line_num, line in enumerate(content.splitlines(), 1):
                    if pattern.lower() in line.lower():
                        results.append(f"{rel_path}:{line_num}: {line.strip()}")
            except Exception:
                continue

        if not results:
            return f"No matches found for pattern: '{pattern}'"

        # Limit results
        max_results = 20
        if len(results) > max_results:
            output = "\n".join(results[:max_results])
            output += f"\n\n... and {len(results) - max_results} more matches"
        else:
            output = "\n".join(results)

        return f"🔍 Found {len(results)} match(es) for '{pattern}':\n\n{output}"

    else:
        return f"❌ Unknown operation: '{operation}'. Use 'list', 'read', or 'grep'."


# ─── Quick Tests ─────────────────────────────────────────────
print("🧪 Testing search_files tool...")

print("\n--- Test 1: List files ---")
print(search_files.invoke({"operation": "list", "path": ""}))

print("\n--- Test 2: Grep for errors ---")
print(search_files.invoke({"operation": "grep", "pattern": "ERROR"}))

print("\n✅ File search tool ready!")

🧪 Testing search_files tool...

--- Test 1: List files ---
📂 root:
  📁 config/ (2 items)
  📁 logs/ (3 items)
  📁 notes/ (2 items)

--- Test 2: Grep for errors ---
🔍 Found 3 match(es) for 'ERROR':

logs/app_2025-05-01.log:5: 2025-05-01 08:45:11 [ERROR] Failed to send email notification: SMTP connection timeout
logs/app_2025-05-02.log:3: 2025-05-02 08:03:22 [ERROR] Database connection pool exhausted. Active connections: 50/50
logs/app_2025-05-02.log:8: 2025-05-02 09:15:33 [ERROR] Payment gateway timeout: Stripe API returned 504

✅ File search tool ready!


# Tool 4 & 5 — Agent Skills (Dynamic Context Injection)

In [24]:
# ===== Tool 4 & 5 — Agent Skills =====
#
# 🧠 THIS IS THE KEY CONCEPT FROM THE WORKSHOP!
#
# "Agent Skills" = dynamically injecting documentation into the LLM's context
# so it can use other tools more effectively.
#
# The Problem:
#   When you ask an agent to query a database, it needs to know:
#   - What tables exist?
#   - What columns does each table have?
#   - What are the relationships?
#   Without this info, the LLM GUESSES → wrong table names → SQL errors
#
# The Solution (Agent Skills):
#   Provide a tool that LOADS the schema into context.
#   The agent calls load_db_schema → gets the schema → writes correct SQL
#
# Same principle applies to API docs, CLI syntax, etc.

# ─── The actual schema content (defined earlier, now stored as a variable) ──

class LoadDBSchemaInput(BaseModel):
    """Input schema for loading database schema."""
    dummy: str = Field(
        default="",
        description="No parameters needed. This tool loads the full database schema."
    )


@tool(args_schema=LoadDBSchemaInput)
def load_db_schema(dummy: str = "") -> str:
    """Load the TechCorp database schema into your context.

    ⚡ CALL THIS TOOL FIRST before writing SQL queries with query_database!

    This tool provides the complete database schema including:
    - All table names and their columns
    - Column data types
    - Foreign key relationships
    - Common query patterns

    Without this schema, you will likely write incorrect SQL queries
    with wrong table names or column names.

    ✅ USE WHEN:
    - You need to write a SQL query but don't know the exact schema
    - A previous query_database call returned an error
    - The user asks a question that requires database knowledge

    ❌ DO NOT USE WHEN:
    - You already have the schema in context (call only once per conversation)
    - The user's question doesn't involve database queries

    Args:
        dummy: No parameters needed.

    Returns:
        The complete database schema with table descriptions.
    """
    logger.info("📖 Agent loaded database schema (Agent Skill)")
    return DB_SCHEMA


class LoadAPIDocsInput(BaseModel):
    """Input schema for loading API documentation."""
    api_name: str = Field(
        default="employee_api",
        description="Name of the API to load docs for. Currently only 'employee_api' is available."
    )


@tool(args_schema=LoadAPIDocsInput)
def load_api_docs(api_name: str = "employee_api") -> str:
    """Load API documentation into your context for writing or understanding API calls.

    This tool provides detailed API documentation including endpoints,
    parameters, request/response formats, and examples.

    ✅ USE WHEN:
    - The user asks about API endpoints or how to call an API
    - You need to construct an API request
    - The user asks about API syntax or parameters

    ❌ DO NOT USE WHEN:
    - The user asks about company policies → use semantic_search instead
    - The user asks about employees/projects → use query_database instead

    Args:
        api_name: Name of the API to load docs for (default: 'employee_api').

    Returns:
        API documentation content.
    """
    docs_path = config.DATA_DIR / "api_docs" / f"{api_name}.md"
    if not docs_path.exists():
        available = [f.stem for f in (config.DATA_DIR / "api_docs").glob("*.md")]
        return f"❌ API docs not found for '{api_name}'. Available: {available}"

    content = docs_path.read_text()
    logger.info(f"📖 Agent loaded API docs for '{api_name}' (Agent Skill)")
    return f"# API Documentation: {api_name}\n\n{content}"


# ─── Quick Tests ─────────────────────────────────────────────
print("🧪 Testing Agent Skills tools...")

print("\n--- Test: load_db_schema ---")
schema_result = load_db_schema.invoke({"dummy": ""})
print(schema_result[:300] + "...")

print("\n--- Test: load_api_docs ---")
api_result = load_api_docs.invoke({"api_name": "employee_api"})
print(api_result[:300] + "...")

print("\n✅ Agent Skills tools ready!")

🧪 Testing Agent Skills tools...

--- Test: load_db_schema ---

TechCorp Database Schema:

Table: departments
  - id (INTEGER, PRIMARY KEY)
  - name (TEXT) — Department name: Engineering, Marketing, Sales, Human Resources, Data Science
  - budget (REAL) — Annual budget in USD
  - head_count (INTEGER) — Number of employees

Table: employees
  - id (INTEGER, PRIM...

--- Test: load_api_docs ---
# API Documentation: employee_api

# Employee API Documentation (v2)

## Base URL
`/api/v2/employees`

## Endpoints

### List Employees
`GET /api/v2/employees`

Query Parameters:
- `department` (string, optional): Filter by department name
- `role` (string, optional): Filter by role title
- `min_sal...

✅ Agent Skills tools ready!


## 🤖 Section 3: Building the Agentic Search Stack

### What is a ReAct Agent?

**ReAct** = **Re**asoning + **Act**ing

The agent follows this loop:
1. **Observe** the user's question
2. **Think** about what tool to use (reasoning)
3. **Act** by calling a tool (acting)
4. **Observe** the tool's result
5. Repeat until it can answer the question

```text
User: "Who is the highest paid engineer and what is our remote work policy?"
 │
 ▼
Agent thinks: "I need two pieces of info. Let me query the database first."
 │
 ├──→ Calls query_database("SELECT name, salary FROM employees
 │                         WHERE department='Engineering' ORDER BY salary DESC LIMIT 1")
 │    Returns: "Jack Thomas, $170,000"
 │
 ▼
Agent thinks: "Now I need the remote work policy from the wiki."
 │
 ├──→ Calls semantic_search("remote work policy")
 │    Returns: "Employees may work remotely up to 3 days per week..."
 │
 ▼
Agent: "The highest paid engineer is Jack Thomas ($170K).
        The remote work policy allows up to 3 days/week..."
```

### Why This Beats Fixed RAG

| Fixed RAG Pipeline | Agentic Search |
|-------------------|----------------|
| Always retrieves (even when unnecessary) | Skips retrieval if it already knows |
| Uses one source only | Picks the right source |
| Cannot combine sources | Combines multiple tools naturally |
| Same search every time | Adapts search strategy per query |
| No error recovery | Can retry with different parameters |

In [25]:
# ===== Build the Agentic Search Agent =====
#
# We use LangGraph's create_react_agent — a production-ready ReAct agent
# that handles the think→act→observe loop automatically.

# ─── Define all tools ────────────────────────────────────────
all_tools = [
    semantic_search,     # Search wiki/docs
    query_database,      # SQL queries
    search_files,        # File system access
    load_db_schema,      # Agent Skill: DB schema
    load_api_docs,       # Agent Skill: API docs
]

# ─── System Prompt (the "brain" of the agent) ────────────────
# This is where we set the agent's personality and decision-making rules.
# A good system prompt is like a good job description — it tells the agent
# exactly what's expected.

SYSTEM_PROMPT = """You are a helpful AI assistant for TechCorp employees. Your job is to answer
questions by searching across the company's knowledge sources.

## Your Available Tools

You have access to the following search tools:

1. **semantic_search** — Search company wiki/policies
   - Use for: policies, procedures, standards, general company knowledge
   - Example questions: "What's the remote work policy?", "How to report a security incident?"

2. **query_database** — Execute SQL on the employee/project database
   - Use for: employee info, departments, projects, salaries, aggregations
   - ⚠️ ALWAYS call load_db_schema FIRST before writing SQL queries!
   - Example questions: "Who earns the most?", "How many projects are active?"

3. **search_files** — Search log files, configs, and meeting notes
   - Use for: logs, configs, meeting notes, file contents
   - Example questions: "Check for errors in the logs", "What's in the app config?"

4. **load_db_schema** — Load database schema (Agent Skill)
   - Call this BEFORE using query_database if you don't already have the schema in context

5. **load_api_docs** — Load API documentation (Agent Skill)
   - Call this when the user asks about API endpoints or how to use an API

## Decision Rules

1. **Think before you act**: Consider which tool(s) you need before calling them.
2. **One tool at a time**: Call one tool, observe the result, then decide next step.
3. **Don't over-retrieve**: If you can answer from the current context, don't call more tools.
4. **Combine sources**: Some questions need multiple tools. That's expected.
5. **Be honest**: If no tool has the answer, say so. Don't hallucinate.
6. **Be concise**: Give direct answers with the relevant information. Don't dump raw data.
"""

# ─── Create the Agent ────────────────────────────────────────
agent = create_react_agent(
    model=llm,
    tools=all_tools,
    prompt=SYSTEM_PROMPT,
)

print("✅ Agentic Search Agent created!")
print(f"   Tools: {[t.name for t in all_tools]}")
print(f"   Model: {config.DEEPSEEK_MODEL}")
print("\n🚀 Ready to answer questions!")

# ─── Helper function to run queries ──────────────────────────
def ask_agent(question: str, verbose: bool = True) -> str:
    """Run a question through the agent and return the answer.

    Args:
        question: The user's question.
        verbose: If True, print intermediate steps (tool calls and results).

    Returns:
        The agent's final answer.
    """
    print(f"\n{'='*60}")
    print(f"🙋 Question: {question}")
    print(f"{'='*60}\n")

    result = agent.invoke(
        {"messages": [HumanMessage(content=question)]}
    )

    # Extract and display the conversation
    final_answer = ""
    for msg in result["messages"]:
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            for tc in msg.tool_calls:
                if verbose:
                    print(f"  🔧 Tool Call: {tc['name']}({json.dumps(tc['args'], indent=2)})")
        elif hasattr(msg, "name") and hasattr(msg, "content"):
            # Tool result
            if verbose:
                preview = msg.content[:200].replace("\n", " ")
                print(f"  📋 Tool Result ({msg.name}): {preview}...")
        elif isinstance(msg, HumanMessage):
            continue
        elif isinstance(msg, type(result["messages"][-1])):
            final_answer = msg.content

    # The last message should be the AI's final answer
    final_msg = result["messages"][-1]
    if hasattr(final_msg, "content"):
        final_answer = final_msg.content

    print(f"\n🤖 Answer:\n{final_answer}")
    return final_answer

✅ Agentic Search Agent created!
   Tools: ['semantic_search', 'query_database', 'search_files', 'load_db_schema', 'load_api_docs']
   Model: deepseek-chat

🚀 Ready to answer questions!


/tmp/ipykernel_58528/1340944377.py:57: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


# Code (Demo 1)

In [26]:
# ===== Demo 1 — Simple Single-Tool Queries =====
# Each of these questions should trigger exactly ONE tool.

# ─── Query 1: Semantic Search ────────────────────────────────
# This should trigger semantic_search (company policy question)
ask_agent("What is the company's remote work policy?")

# ─── Query 2: Database Query ─────────────────────────────────
# This should trigger load_db_schema → query_database
ask_agent("Who are all the employees in the Engineering department and what are their salaries?")

# ─── Query 3: File Search ────────────────────────────────────
# This should trigger search_files
ask_agent("Are there any errors in the application logs from May 2nd?")


🙋 Question: What is the company's remote work policy?

  📋 Tool Result (None): What is the company's remote work policy?...
  🔧 Tool Call: semantic_search({
  "query": "remote work policy",
  "top_k": 3
})
  📋 Tool Result (semantic_search): 📄 Result 1 [Remote Work Policy]: TechCorp Remote Work Policy (Updated 2025)  Eligibility: All full-time employees who have completed their 90-day probation period are eligible for remote work.  Schedu...
  📋 Tool Result (None): Here's a summary of TechCorp's **Remote Work Policy** (Updated 2025):  ### Eligibility - All **full-time employees** who have completed their **90-day probation period** are eligible.  ### Schedule - ...

🤖 Answer:
Here's a summary of TechCorp's **Remote Work Policy** (Updated 2025):

### Eligibility
- All **full-time employees** who have completed their **90-day probation period** are eligible.

### Schedule
- Work remotely **up to 3 days per week**
- At least **2 days in-office** for team collaboration
- Most popular remot

"Yes, there were **two errors** in the application logs from May 2nd. Here's a summary:\n\n---\n\n### 🔴 Error 1: Database Connection Pool Exhausted\n- **Time:** 08:03:22\n- **Severity:** CRITICAL\n- **Cause:** Too many concurrent requests (~200 employees) during the morning login rush\n- **Impact:** The connection pool hit its 50/50 limit, causing requests to queue\n- **Resolution:** Pool size was increased from **50 → 75**, and a 30-second connection timeout was added\n- **Status:** ✅ Recovered by 08:05:44\n\n### 🔴 Error 2: Payment Gateway Timeout (Stripe)\n- **Time:** 09:15:33\n- **Severity:** CRITICAL\n- **Cause:** Stripe API returned a 504 (degradation incident #INC-44521)\n- **Impact:** 3 payments totaling **$12,450** were affected\n- **Resolution:** Exponential backoff retry was implemented, and all payments were successfully processed by 09:20:11\n- **Status:** ✅ All payments recovered\n\nBoth issues were resolved on the same day with no lasting impact."

# Code (Demo 2)

In [27]:
# ===== Demo 2 — Complex Multi-Tool Queries =====
# These questions require the agent to combine MULTIPLE tools.
# This is where Agentic Search truly shines vs. fixed RAG pipelines.

# ─── Query 4: Multi-source (DB + Wiki) ───────────────────────
# The agent needs both employee data AND company policy
ask_agent(
    "Jack Thomas wants to apply for a sabbatical. "
    "What is his role and how long has he been at the company, "
    "and what is the sabbatical policy?"
)

# ─── Query 5: Multi-source (Files + DB) ──────────────────────
# The agent needs log info AND context about who's responsible
ask_agent(
    "There was a database connection pool issue recently. "
    "Find the details in the logs and tell me who the Security Engineer is that should be notified."
)


🙋 Question: Jack Thomas wants to apply for a sabbatical. What is his role and how long has he been at the company, and what is the sabbatical policy?

  📋 Tool Result (None): Jack Thomas wants to apply for a sabbatical. What is his role and how long has he been at the company, and what is the sabbatical policy?...
  🔧 Tool Call: load_db_schema({})
  🔧 Tool Call: semantic_search({
  "query": "sabbatical policy",
  "top_k": 3
})
  📋 Tool Result (load_db_schema):  TechCorp Database Schema:  Table: departments   - id (INTEGER, PRIMARY KEY)   - name (TEXT) — Department name: Engineering, Marketing, Sales, Human Resources, Data Science   - budget (REAL) — Annual ...
  📋 Tool Result (semantic_search): 📄 Result 1 [Leave and PTO Policy]: TechCorp Leave and PTO Policy (Updated 2025)  Annual Leave: Full-time employees receive 20 PTO days per year, accrued at 1.67 days per month. PTO rolls over up to 5 ...
  🔧 Tool Call: query_database({
  "sql_query": "SELECT * FROM employees WHERE name = 'Jack 

"Here's a full summary of what I found:\n\n---\n\n## 🔍 Database Connection Pool Exhaustion Incident\n\n### Incident Details\n- **Date**: May 2, 2025 (also referenced in a postmortem from April 28, 2025)\n- **Time**: 8:03 AM UTC\n- **Severity**: P2 (Critical)\n- **Duration**: ~2 hours 15 minutes\n- **Impact**: ~200 employees unable to access the employee portal during the morning login rush\n\n### Root Cause\n- The connection pool (50 connections) was exhausted by 3x normal traffic during the morning rush\n- No connection timeout was configured, so idle connections were never released\n- Monitoring alert threshold was set too high at 90%\n\n### Fixes Applied\n1. ✅ Pool size increased from **50 → 75** connections\n2. ✅ Added **30-second connection timeout**\n3. ✅ Monitoring alert threshold lowered to **70%**\n4. ✅ Connection pool metrics dashboard implemented\n\n---\n\n## 👤 Security Engineer to Notify\n\n**Leo White**\n- **Role**: Security Engineer\n- **Department**: Engineering\n- **Ema

# Code (Demo 3)

In [28]:
# ===== Demo 3 — When NOT to Search =====
# A good agent knows when it DOESN'T need to search.
# Simple questions that don't require company-specific data
# should be answered directly.

ask_agent("What is 2 + 2?")


🙋 Question: What is 2 + 2?

  📋 Tool Result (None): What is 2 + 2?...
  📋 Tool Result (None): That's a simple math question — no need for any tools!  2 + 2 = 4....

🤖 Answer:
That's a simple math question — no need for any tools!

2 + 2 = 4.


"That's a simple math question — no need for any tools!\n\n2 + 2 = 4."

# Code (Failure Mode Demo)

In [29]:
# ==== Demo 4 — Failure Mode: Bad Tool Descriptions =====
#
# 🧠 KEY WORKSHOP INSIGHT: Bad tool descriptions → wrong tool selection

print("""
╔══════════════════════════════════════════════════════════════╗
║  🔴 FAILURE MODE DEMONSTRATION: Bad Tool Descriptions        ║
╚══════════════════════════════════════════════════════════════╝

We'll create tools with MINIMAL descriptions (a common mistake)
and show how the agent picks the wrong tool.
""")

# ─── Badly-described tools ───────────────────────────────────

@tool
def search_bad(query: str) -> str:
    """Search for information."""  # ❌ Vague! No trigger conditions, no boundaries
    results = vectorstore.similarity_search(query, k=3)
    if not results:
        return "No results found."
    return "\n\n---\n\n".join(f"[{d.metadata['title']}]: {d.page_content}" for d in results)

@tool
def query_db_bad(query: str) -> str:  # ❌ Parameter named "query" not "sql_query" — ambiguous!
    """Query the database."""  # ❌ No schema info, no trigger conditions
    try:
        conn = sqlite3.connect(str(config.DB_PATH))
        cursor = conn.cursor()
        cursor.execute(query)
        columns = [d[0] for d in cursor.description]
        rows = cursor.fetchall()
        conn.close()
        return "\n".join(" | ".join(str(v) for v in row) for row in rows)
    except Exception as e:
        return f"Error: {e}"

bad_tools = [search_bad, query_db_bad]

bad_agent = create_react_agent(
    model=llm,
    tools=bad_tools,
    prompt="You are a helpful assistant. Answer questions using the available tools.",
)

print("🔴 Testing with BAD tool descriptions...")
print("   Question: 'How many employees are in each department?'")
print("   Expected: Should use query_db_bad (SQL query)")
print("   What might happen: Agent uses search_bad (semantic) instead")
print("   because it doesn't know when to use which tool!\n")

result = bad_agent.invoke({
    "messages": [HumanMessage(content="How many employees are in each department?")]
})

for msg in result["messages"]:
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"  🔧 Tool Called: {tc['name']} with args: {tc['args']}")
    elif hasattr(msg, "name"):
        print(f"  📋 Tool Result: {msg.content[:150]}...")

final = result["messages"][-1].content
print(f"\n🤖 Agent Answer:\n{final[:500]}")

print("""
╔══════════════════════════════════════════════════════════════╗
║  ✅ COMPARISON: Good vs. Bad Tool Descriptions               ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  ❌ BAD: "Search for information."                           ║
║  ✅ GOOD: "Search company wiki using semantic similarity.    ║
║           Use for policies, procedures.                      ║
║           NOT for structured data (use query_database)."     ║
║                                                              ║
║  ❌ BAD: "Query the database."                               ║
║  ✅ GOOD: "Execute SQL on employee/project database.         ║
║           Use for employees, departments, projects.          ║
║           Call load_db_schema first for schema."             ║
║                                                              ║
║  The difference is:                                          ║
║  1. WHAT the tool does (specific, not vague)                 ║
║  2. WHEN to use it (trigger conditions)                      ║
║  3. WHEN NOT to use it (boundaries)                          ║
║  4. HOW to use it (parameter guidance)                       ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝
""")


╔══════════════════════════════════════════════════════════════╗
║  🔴 FAILURE MODE DEMONSTRATION: Bad Tool Descriptions        ║
╚══════════════════════════════════════════════════════════════╝

We'll create tools with MINIMAL descriptions (a common mistake)
and show how the agent picks the wrong tool.

🔴 Testing with BAD tool descriptions...
   Question: 'How many employees are in each department?'
   Expected: Should use query_db_bad (SQL query)
   What might happen: Agent uses search_bad (semantic) instead
   because it doesn't know when to use which tool!



/tmp/ipykernel_58528/1953040823.py:40: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  bad_agent = create_react_agent(


  📋 Tool Result: How many employees are in each department?...
  🔧 Tool Called: search_bad with args: {'query': 'employees per department'}
  📋 Tool Result: [Leave and PTO Policy]: TechCorp Leave and PTO Policy (Updated 2025)

Annual Leave: Full-time employees receive 20 PTO days per year, accrued at 1.67 ...
  🔧 Tool Called: query_db_bad with args: {'query': 'SELECT department, COUNT(*) as employee_count FROM employees GROUP BY department'}
  📋 Tool Result: Data Science | 2
Engineering | 5
Human Resources | 1
Marketing | 2
Sales | 2...
  📋 Tool Result: Here is the number of employees in each department:

| Department | Employees |
|------------|-----------|
| **Data Science** | 2 |
| **Engineering** ...

🤖 Agent Answer:
Here is the number of employees in each department:

| Department | Employees |
|------------|-----------|
| **Data Science** | 2 |
| **Engineering** | 5 |
| **Human Resources** | 1 |
| **Marketing** | 2 |
| **Sales** | 2 |

**Total: 12 employees** across all departmen

## 📊 Section 4: Practical Recommendations

### The Workshop's Key Takeaways

Based on Leonie Monigatti's recommendations:

### 1. 🏗️ Build a Search Tool Stack (Not a Single Tool)

```text
                    ┌─────────────┐
                    │  User Query │
                    └──────┬──────┘
                           │
                    ┌──────▼──────┐
                    │    Agent    │
                    └──┬───┬──┬───┘
                       │   │  │
         ┌─────────────┘   │  └─────────────┐
         ▼                 ▼                ▼
  ┌─────────────┐   ┌─────────────┐  ┌─────────────┐
  │  Semantic   │   │  Database   │  │   Shell /   │
  │   Search    │   │   Query     │  │    Files    │
  │  (Common)   │   │ (Structured │  │   (Deep     │
  │             │   │    Data)    │  │    Dive)    │
  └─────────────┘   └─────────────┘  └─────────────┘
   Low Floor ✅       + Agent Skills   High Ceiling 🚀
   Easy queries       for accuracy     Complex queries
```

**"Low Floor, High Ceiling"** — Make common queries easy (semantic search),
but support complex queries too (shell, SQL, custom CLIs).

### 2. 📝 Write Detailed Tool Descriptions

| Element | Why It Matters | Example |
|---------|---------------|---------|
| **What** | Agent knows the tool's purpose | "Search wiki using semantic similarity" |
| **When** | Agent knows trigger conditions | "Use for policies, procedures" |
| **When NOT** | Agent avoids wrong tool | "NOT for structured data" |
| **Parameters** | Agent generates correct args | "Natural language query, not SQL" |
| **Relationships** | Agent knows tool boundaries | "For DB queries, use query_database instead" |

### 3. 🧠 Use Agent Skills for Complex Tools

When a tool requires specialized knowledge (SQL syntax, API endpoints, CLI flags):
- **Don't** hardcode that knowledge in the system prompt (wastes tokens on every call)
- **Do** create a "Skill" tool that loads the knowledge on demand
- The agent calls the skill tool when needed, then uses the knowledge

### 4. 🔒 Sandbox the Shell Tool

The shell tool is the most versatile but most dangerous:
- **Never** give raw shell access in production
- **Do** provide specific operations (list, read, grep) instead of arbitrary commands
- **Do** restrict paths and validate inputs
- **Do** run in a container/sandbox in production

### 5. 🔄 Handle Failure Modes Gracefully

| Failure Mode | Cause | Fix |
|-------------|-------|-----|
| Agent doesn't call any tool | Doesn't know a tool exists | Better tool descriptions |
| Agent calls wrong tool | Vague descriptions | Add "USE FOR" / "DON'T USE FOR" |
| Agent generates wrong params | Unclear parameter docs | Add examples and constraints |
| Agent calls too many tools | No guidance on when to stop | Add "call only once" instructions |

### 6. 📈 Monitor and Iterate

In production, track:
- Which tools are called most/least?
- What queries fail?
- What parameters are wrong?
- Are there queries that need a new tool?

Use this data to improve tool descriptions and add new tools.

In [31]:
# ===== Interactive Playground =====
#
# Try your own questions here!
# The agent will decide which tool(s) to use.

def interactive_query(question: str):
    """Ask the agent a question and see the full reasoning process."""
    return ask_agent(question, verbose=True)

# ─── Example questions to try ────────────────────────────────
# Uncomment any of these, or write your own!

# interactive_query("What is the PTO policy and how many vacation days do I get?")
# interactive_query("Show me all projects that are currently in progress and their budgets")
# interactive_query("Find any mentions of 'timeout' in the logs and explain what happened")
# interactive_query("What API endpoints are available for the Employee API?")
# interactive_query("Who is the newest hire in Data Science and what is the onboarding process?")
# interactive_query("What was discussed in the Q2 planning meeting about the CloudGuard project?")
# interactive_query("What's the average salary across all departments?")
# interactive_query("How should I report a P1 security incident?")

# Try your own question:
interactive_query("What's the total budget for all in-progress projects, and what does the security incident response plan say about P1 incidents?")


🙋 Question: What's the total budget for all in-progress projects, and what does the security incident response plan say about P1 incidents?

  📋 Tool Result (None): What's the total budget for all in-progress projects, and what does the security incident response plan say about P1 incidents?...
  🔧 Tool Call: load_db_schema({})
  🔧 Tool Call: semantic_search({
  "query": "security incident response plan P1 incidents",
  "top_k": 3
})
  📋 Tool Result (load_db_schema):  TechCorp Database Schema:  Table: departments   - id (INTEGER, PRIMARY KEY)   - name (TEXT) — Department name: Engineering, Marketing, Sales, Human Resources, Data Science   - budget (REAL) — Annual ...
  📋 Tool Result (semantic_search): 📄 Result 1 [Security Incident Response]: TechCorp Security Incident Response Plan (v2.1)  Severity Levels: - P1 (Critical): Data breach, ransomware, complete service outage. Response time: 15 minutes....
  🔧 Tool Call: query_database({
  "sql_query": "SELECT SUM(budget) AS total_budget F

'Here are the answers to both questions:\n\n---\n\n### 1. Total Budget for In-Progress Projects\nThe total budget for all projects with a status of **"In Progress"** is **$1,350,000**.\n\n---\n\n### 2. Security Incident Response Plan — P1 (Critical) Incidents\nAccording to the **TechCorp Security Incident Response Plan (v2.1)**:\n\n- **Definition**: P1 (Critical) incidents include **data breaches, ransomware, and complete service outages**.\n- **Response Time**: Must be triaged within **15 minutes**.\n- **Escalation**: For P1/P2 incidents, an **Incident Commander is assigned within 30 minutes**.\n- **Reporting**: Report to `security@techcorp.com` or the `#security-incidents` Slack channel.\n- **Post-Incident**: A **blameless retrospective** is held within **5 business days**, with action items tracked in Jira (30-day completion deadline).\n\n**Key Contacts**:\n- **CISO**: Sarah Chen (sarah.chen@techcorp.com)\n- **Security Lead**: Marcus Rivera (marcus.rivera@techcorp.com)\n- **On-call*

# FINAL SUMMARY

## Here's a summary of every key concept:

### 📖 Concept Explanations

| Concept | Simple Explanation |
|---------|-------------------|
| **Context Engineering** | Choosing *what information* to feed the LLM. Like giving a student the right textbook before an exam — if you give them the wrong book, even a genius fails. |
| **Agentic Search** | Instead of always searching (naive RAG), the *agent decides* whether and where to search. Like a librarian who knows which section to look in. |
| **Semantic Search** | Searches by *meaning*, not keywords. "Work from home" matches "remote work policy" because they mean the same thing. Uses vector embeddings (numbers that represent meaning). |
| **Vector Embeddings** | Text → array of numbers. Similar text → similar numbers. Like GPS coordinates for meaning. |
| **ChromaDB** | A vector database that stores embeddings and finds the closest matches. Like a phone book for meanings. |
| **Tool Descriptions** | The text that tells the LLM what a tool does. Like a job description — if it's vague, the wrong person applies. |
| **Agent Skills** | Dynamically loading documentation (like a DB schema) into the LLM's context before it uses a tool. Like giving someone a map before asking for directions. |
| **ReAct Agent** | An agent that alternates between **Re**asoning (thinking) and **Act**ing (calling tools). Like a detective: think → investigate → think more → solve. |
| **Shell Tool** | A tool that lets the agent interact with the file system. Very powerful but dangerous — must be sandboxed. Like giving someone keys to the building. |
| **Failure Modes** | The 4 ways agents fail: (1) don't call any tool, (2) call wrong tool, (3) wrong parameters, (4) call too many tools. |
| **Sandboxing** | Restricting what a tool can do so the agent can't cause harm. Like giving read-only access instead of admin access. |
| **Low Floor, High Ceiling** | Design philosophy: make common tasks easy (semantic search = low floor), but support complex tasks too (shell/SQL = high ceiling). |



